# PP-OCRv5 한국어 인식 모델 → ONNX 변환 (Jetson Nano용)

PP-OCRv3가 못 읽던 `154러7070`을 PP-OCRv5 한국어 모델은 신뢰도 0.98로 읽었습니다.
이 모델을 Jetson(onnxruntime 1.11)에서 쓸 수 있도록 변환합니다.

| 만들어지는 파일 | 용도 |
|---|---|
| `models_v5/rec.onnx` | PP-OCRv5 한국어 인식 모델 |
| `models_v5/korean_dict.txt` | 이 모델 전용 글자 사전 (**v3 사전과 다름, 섞어 쓰면 안 됨**) |

v3 때와 다른 점:
- v5 모델은 새 저장 형식(`inference.json`)이라 새 변환 도구(paddle2onnx 2.x + paddle 3.x)가 필요
- 글자 사전이 별도 파일이 아니라 모델 설정 파일(`inference.yml`) 안에 들어 있어서 꺼내야 함

런타임은 CPU로 충분합니다. 위에서부터 순서대로 실행하세요.

## 1. 모델 받기 + 원본(Paddle)으로 한 번 더 확인

`debug_2_line.png`(Jetson에서 저장된 글자 줄 이미지)를 왼쪽 파일 탭에 올려두면 원본 모델 결과도 출력합니다.
이 결과가 5번 셀에서 변환된 모델의 정답 기준이 됩니다.

In [ ]:
!pip install -q paddlepaddle paddleocr

import os
from paddleocr import TextRecognition

rec = TextRecognition(model_name="korean_PP-OCRv5_mobile_rec", enable_mkldnn=False)
MODEL_DIR = "/root/.paddlex/official_models/korean_PP-OCRv5_mobile_rec"
print("모델 폴더:", os.listdir(MODEL_DIR))

if os.path.exists("/content/debug_2_line.png"):
    for res in rec.predict("/content/debug_2_line.png"):
        print("원본 모델 결과:", res["rec_text"], round(float(res["rec_score"]), 4))

## 2. 변환용 격리 환경

Colab 기본 Python은 버전이 높아서, Jetson과 같은 onnxruntime 1.11을 설치할 수 있는 Python 3.9 환경을 따로 만듭니다.
설치 중 `onnx_graphsurgeon` 관련 설치 오류 문구가 나와도 변환에는 영향이 없습니다.

In [ ]:
!pip install -q uv
!uv venv --python 3.9 /content/v5env
!uv pip install --python /content/v5env/bin/python \
    "paddlepaddle>=3.0,<3.3" "paddle2onnx>=2.0" "onnx<1.17" "onnxruntime==1.11.1" \
    "numpy<2" "opencv-python-headless<4.11" setuptools packaging pyyaml

!/content/v5env/bin/python -c "import paddle, paddle2onnx, onnxruntime as ort; \
print('paddle', paddle.__version__, '/ paddle2onnx', paddle2onnx.__version__, '/ onnxruntime', ort.__version__)"

## 3. ONNX 변환

`opset_version 11`: Jetson의 onnxruntime 1.11이 확실히 지원하는 버전으로 고정

In [ ]:
!mkdir -p /content/models_v5
!/content/v5env/bin/paddle2onnx \
    --model_dir /root/.paddlex/official_models/korean_PP-OCRv5_mobile_rec \
    --model_filename inference.json \
    --params_filename inference.pdiparams \
    --save_file /content/models_v5/rec.onnx \
    --opset_version 11 \
    --enable_onnx_checker True
!ls -lh /content/models_v5

## 4. 글자 사전 꺼내기

모델 설정 파일(`inference.yml`)의 `character_dict`를 한 줄에 한 글자씩 저장합니다.

In [ ]:
%%writefile /content/extract_dict.py
import sys
import yaml

MODEL_DIR = "/root/.paddlex/official_models/korean_PP-OCRv5_mobile_rec"
OUT = "/content/models_v5/korean_dict.txt"

def find(node, key):
    """설정 파일 어디에 있든 key를 찾아 값을 반환"""
    if isinstance(node, dict):
        if key in node:
            return node[key]
        for v in node.values():
            r = find(v, key)
            if r is not None:
                return r
    elif isinstance(node, list):
        for v in node:
            r = find(v, key)
            if r is not None:
                return r
    return None

with open(MODEL_DIR + "/inference.yml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

chars = find(cfg, "character_dict")
if not chars:
    print("[실패] character_dict를 찾지 못함. 최상위 항목:", list(cfg.keys()))
    sys.exit(1)

with open(OUT, "w", encoding="utf-8") as f:
    f.write("\n".join(str(c) for c in chars) + "\n")

print("사전 {}자 저장 -> {}".format(len(chars), OUT))
print("입력 크기 설정:", find(cfg, "image_shape"))
missing = [c for c in "가나다라마거너더러머버서어저고노도로모보소오조구누두루무부수우주아바사자허하호배"
           if c not in chars]
print("번호판 한글 누락:", "".join(missing) if missing else "없음")

In [ ]:
!/content/v5env/bin/python /content/extract_dict.py

## 5. Jetson과 같은 onnxruntime 1.11로 검증

Jetson의 `plate_text.py`와 같은 전처리·해석 방식으로 `debug_2_line.png`를 읽습니다.
**1번 셀의 원본 모델 결과와 같게 나오면** 변환 성공입니다.

In [ ]:
%%writefile /content/verify.py
import math, os, sys
from itertools import groupby
import cv2
import numpy as np
import onnx
import onnxruntime as ort

MODEL = "/content/models_v5/rec.onnx"
DICT = "/content/models_v5/korean_dict.txt"
IMAGE = "/content/debug_2_line.png"

# [1] 모델 형식: onnxruntime 1.11은 IR 8까지만 읽음
m = onnx.load(MODEL)
opset = max(o.version for o in m.opset_import if o.domain in ("", "ai.onnx"))
print("[1] 모델 형식  IR {}, opset {}".format(m.ir_version, opset))
if m.ir_version > 8:
    m.ir_version = 8
    onnx.save(m, MODEL)
    print("    -> IR 8로 낮춰 다시 저장")

# [2] 실행 + 사전 대조
sess = ort.InferenceSession(MODEL, providers=["CPUExecutionProvider"])
inp = sess.get_inputs()[0]
with open(DICT, encoding="utf-8") as f:
    charset = ["<blank>"] + [l.rstrip("\r\n") for l in f] + [" "]
out = sess.run(None, {inp.name: np.zeros((1, 3, 48, 320), np.float32)})[0]
if out.shape[-1] == len(charset) - 1:
    charset.pop()
ok = out.shape[-1] == len(charset)
print("[2] 실행       입력 {} -> 출력 {} / 사전 {} -> {}".format(
    inp.shape, out.shape, len(charset), "OK" if ok else "불일치"))

# [3] 실제 이미지 인식 (Jetson plate_text.py와 같은 전처리)
if not os.path.exists(IMAGE):
    print("[3] debug_2_line.png 없음 -> 이미지 인식 확인 생략")
    sys.exit(0)
img = cv2.imread(IMAGE)
h, w = img.shape[:2]
rw = int(math.ceil(48 * w / float(h)))
tw = max(320, rw)
x = np.zeros((1, 3, 48, tw), np.float32)
x[0, :, :, :rw] = ((cv2.resize(img, (rw, 48)).astype(np.float32) / 255.0 - 0.5) / 0.5).transpose(2, 0, 1)
probs = sess.run(None, {inp.name: x})[0][0]
steps = zip(probs.argmax(axis=1), probs.max(axis=1))
chars = [(charset[i], float(next(g)[1])) for i, g in groupby(steps, key=lambda s: s[0]) if i != 0]
print("[3] 인식 결과  '{}' (평균 확률 {:.3f})".format(
    "".join(c for c, _ in chars), np.mean([p for _, p in chars]) if chars else 0))

In [ ]:
!cd /content && /content/v5env/bin/python verify.py 2>&1 | grep -v "W:onnxruntime"

## 6. 다운로드

`models_v5.zip`을 Jetson의 `~/smartparking/Models/paddleocr/`에 풀면 `models_v5/` 폴더가 생깁니다.
기존 v3 모델(`models/`)은 비교용으로 그대로 두세요.

In [ ]:
%cd /content
!rm -f models_v5.zip && zip -r models_v5.zip models_v5/rec.onnx models_v5/korean_dict.txt
from google.colab import files
files.download("/content/models_v5.zip")